# Validação dos Modelos MTL — Comparação com Benchmarks

Este notebook consolida:
- **Benchmarks** (Random Forest ST, Ridge, Dummy) × MTL por split e via
- **Estatísticas de validação** (R², MAE, CCC, IC 95% bootstrap, Wilcoxon p-value)
- **Tabelas de publicação** formatadas

> **Splits disponíveis:** `random`, `butina`, `scaffold`  
> **Vias:** `mouse_vi`, `mouse_vo`, `mouse_ip`, `rat_vi`, `rat_vo`, `rat_ip`

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Raiz do projeto
current_dir = Path(globals()['_dh'][0]).resolve()
RAIZ = current_dir.parent.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

print(f'Raiz do projeto: {RAIZ}')

In [ ]:
# ── Caminhos ──────────────────────────────────────────────────
PATHS = {
    'tabelas':           RAIZ / 'results' / 'tabelas' / 'multitask',
    'tabelas_validacao': RAIZ / 'results' / 'tabelas' / 'multitask' / 'validacao',
    'plots_validacao':   RAIZ / 'results' / 'plots'   / 'multitask' / 'validacao',
    'melhor_modelo':     RAIZ / 'results' / 'melhor_modelo',
}

SPLITS = ['random', 'butina', 'scaffold']
VIAS   = ['mouse_vi', 'mouse_vo', 'mouse_ip', 'rat_vi', 'rat_vo', 'rat_ip']
VIAS_LABEL = {
    'mouse_vi': 'Mouse IV',
    'mouse_vo': 'Mouse VO',
    'mouse_ip': 'Mouse IP',
    'rat_vi':   'Rat IV',
    'rat_vo':   'Rat VO',
    'rat_ip':   'Rat IP',
}

SPLIT_LABEL = {'random': 'Random', 'butina': 'Butina', 'scaffold': 'Scaffold'}

# Estética dos gráficos
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'axes.facecolor':   '#F9F9F9',
    'figure.facecolor': 'white',
    'axes.grid':        True,
    'grid.color':       '#E0E0E0',
    'grid.linestyle':   '--',
    'axes.axisbelow':   True,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

CORES_MODELO = {
    'MTL (nosso)':        '#2196F3',
    'Random Forest (ST)': '#4CAF50',
    'Ridge Regression':   '#FF9800',
    'Dummy (mean)':       '#9E9E9E',
}

print('Setup concluído.')

## 1. Carregamento dos Dados

In [ ]:
# ── 1A. Resumo dos melhores modelos ───────────────────────────
df_resumo = pd.read_excel(PATHS['melhor_modelo'] / 'resumo_melhores_modelos.xlsx')
print('Colunas disponíveis no resumo:', list(df_resumo.columns))
df_resumo

In [ ]:
# ── 1B. Tabelas estatísticas (Resumo_Publicacao_ABNT) por split ──
stats_mtl: dict[str, pd.DataFrame] = {}

for split in SPLITS:
    path = PATHS['tabelas_validacao'] / split / f'Tabela_Estatistica_{split}.xlsx'
    if not path.exists():
        print(f'  [!] Não encontrado: {path}')
        continue

    xl = pd.ExcelFile(path)
    print(f'{split:10s}  |  sheets: {xl.sheet_names}')

    if 'Resumo_Publicacao_ABNT' in xl.sheet_names:
        df = xl.parse('Resumo_Publicacao_ABNT', index_col=0)
        df.index.name = 'Via'
        df['Split'] = split
        stats_mtl[split] = df

print('\nSplits carregados:', list(stats_mtl.keys()))

In [ ]:
# ── 1C. Benchmarks por split ────────────────────────────────
benchmarks: dict[str, pd.DataFrame] = {}

for split in SPLITS:
    path = PATHS['tabelas_validacao'] / split / f'Benchmarks_{split}.xlsx'
    if not path.exists():
        print(f'  [!] Não encontrado: {path}')
        continue
    df = pd.read_excel(path)
    df['Split'] = split
    benchmarks[split] = df
    print(f'{split:10s}  |  modelos: {df["Modelo"].unique().tolist()}')

print('\nBenchmarks carregados:', list(benchmarks.keys()))

In [ ]:
# ── 1D. Construir df_mtl_perf: métricas do MTL no mesmo formato dos benchmarks ──
# Fonte: Resumo_Publicacao_ABNT — colunas R², MAE

registros = []
for split, df in stats_mtl.items():
    for via in df.index:
        row = df.loc[via]
        registros.append({
            'Modelo': 'MTL (nosso)',
            'Via':    via,
            'R2':     row.get('R²',  np.nan),
            'MAE':    row.get('MAE', np.nan),
            'CCC':    row.get('CCC', np.nan),
            'IC95_R2':row.get('IC95_R2', ''),
            'Split':  split,
        })

df_mtl_perf = pd.DataFrame(registros)
print(f'MTL performance: {len(df_mtl_perf)} linhas')
df_mtl_perf.head()

In [ ]:
# ── 1E. Concatenar benchmarks em df único ─────────────────────
bench_all = pd.concat(benchmarks.values(), ignore_index=True)

# Unir MTL com benchmarks
df_comp = pd.concat(
    [bench_all[['Modelo', 'Via', 'R2', 'MAE', 'Split']], 
     df_mtl_perf[['Modelo', 'Via', 'R2', 'MAE', 'Split']]],
    ignore_index=True
)

print(f'Total de registros para comparação: {len(df_comp)}')
df_comp['Modelo'].value_counts()

## 2. Tabela de Comparação: MTL vs Benchmarks (por Split)

In [ ]:
# ── Tabela pivotada: R² por (Via × Modelo) para cada split ───
ORDER_MODELOS = ['MTL (nosso)', 'Random Forest (ST)', 'Ridge Regression', 'Dummy (mean)']

def fmt_r2(v):
    try:
        return f'{float(v):.4f}'
    except:
        return str(v)

for split in SPLITS:
    df_s = df_comp[df_comp['Split'] == split].copy()
    pivot = df_s.pivot_table(index='Via', columns='Modelo', values='R2', aggfunc='first')
    # reordenar colunas
    cols = [c for c in ORDER_MODELOS if c in pivot.columns]
    pivot = pivot[cols]
    pivot.index = [VIAS_LABEL.get(v, v) for v in pivot.index]

    print(f'\n═══ R² — Split: {SPLIT_LABEL[split]} ═══')
    display(pivot.map(fmt_r2).style
        .set_caption(f'R² por Via e Modelo — {SPLIT_LABEL[split]}')
        .highlight_max(axis=1, props='background-color: #d4edda; color: #155724; font-weight: bold')
        .highlight_min(axis=1, props='background-color: #f8d7da; color: #721c24')
    )

In [ ]:
# ── Tabela pivotada: MAE ───────────────────────────────────────
for split in SPLITS:
    df_s = df_comp[df_comp['Split'] == split].copy()
    pivot = df_s.pivot_table(index='Via', columns='Modelo', values='MAE', aggfunc='first')
    cols = [c for c in ORDER_MODELOS if c in pivot.columns]
    pivot = pivot[cols]
    pivot.index = [VIAS_LABEL.get(v, v) for v in pivot.index]

    print(f'\n═══ MAE — Split: {SPLIT_LABEL[split]} ═══')
    display(pivot.map(fmt_r2).style
        .set_caption(f'MAE por Via e Modelo — {SPLIT_LABEL[split]}')
        # Para MAE: menor é melhor → min fica verde, max fica vermelho
        .highlight_min(axis=1, props='background-color: #d4edda; color: #155724; font-weight: bold')
        .highlight_max(axis=1, props='background-color: #f8d7da; color: #721c24')
    )

## 3. Tabela de Estatísticas do MTL (por Split)

In [ ]:
# Colunas de interesse para publicação
COLS_PUB = ['R²', 'MAE', 'CCC', 'IC95_R2', 'p_Wilcoxon (adj. BH)', 'Significativo (α=0.05)', 'Melhoria_%']

def fmt_pval(v):
    """Formata p-values: zeros reais como '< 1e-300', NaN como '—'."""
    try:
        f = float(v)
        if np.isnan(f):
            return '—'
        if f == 0.0:
            return '< 1e-300'
        if f < 0.001:
            return f'{f:.2e}'
        return f'{f:.4f}'
    except:
        return str(v)

def fmt_sig(v):
    try:
        return '✓' if bool(v) else '✗'
    except:
        return str(v)

for split in SPLITS:
    if split not in stats_mtl:
        continue
    df = stats_mtl[split].copy()
    df.index = [VIAS_LABEL.get(v, v) for v in df.index]

    # Selecionar colunas existentes
    cols = [c for c in COLS_PUB if c in df.columns]
    df_pub = df[cols].copy()

    # Formatar
    for c in ['R²', 'MAE', 'CCC']:
        if c in df_pub.columns:
            df_pub[c] = df_pub[c].apply(lambda x: f'{float(x):.4f}' if x != '' else '—')
    if 'p_Wilcoxon (adj. BH)' in df_pub.columns:
        df_pub['p_Wilcoxon (adj. BH)'] = df_pub['p_Wilcoxon (adj. BH)'].apply(fmt_pval)
    if 'Significativo (α=0.05)' in df_pub.columns:
        df_pub['Significativo (α=0.05)'] = df_pub['Significativo (α=0.05)'].apply(fmt_sig)
    if 'Melhoria_%' in df_pub.columns:
        df_pub['Melhoria_%'] = df_pub['Melhoria_%'].apply(lambda x: f'{float(x):.1f}%' if x != '' else '—')

    print(f'\n═══ Estatísticas MTL — Split: {SPLIT_LABEL[split]} ═══')
    display(df_pub.style.set_caption(f'Validação Estatística — {SPLIT_LABEL[split]}'))

## 4. Tabela Consolidada (todos os splits)

In [ ]:
# Construir tabela de publicação unificada: Split × Via com R², MAE, CCC, Wilcoxon
registros_pub = []
for split, df in stats_mtl.items():
    for via in df.index:
        row = df.loc[via]
        registros_pub.append({
            'Split':       SPLIT_LABEL[split],
            'Via':         VIAS_LABEL.get(via, via),
            'R² (MTL)':    float(row.get('R²',  np.nan)),
            'MAE (MTL)':   float(row.get('MAE', np.nan)),
            'CCC':         float(row.get('CCC', np.nan)),
            'IC95 R²':     str(row.get('IC95_R2', '—')),
            'p-Wilcoxon':  row.get('p_Wilcoxon (adj. BH)', np.nan),
            'Sign.':       row.get('Significativo (α=0.05)', np.nan),
            'Δ MAE %':     float(row.get('Melhoria_%', np.nan)),
        })

df_pub_all = pd.DataFrame(registros_pub)

# Formatar para exibição
df_disp = df_pub_all.copy()
df_disp['R² (MTL)'] = df_disp['R² (MTL)'].apply(lambda x: f'{x:.4f}')
df_disp['MAE (MTL)']= df_disp['MAE (MTL)'].apply(lambda x: f'{x:.4f}')
df_disp['CCC']      = df_disp['CCC'].apply(lambda x: f'{x:.4f}')
df_disp['p-Wilcoxon'] = df_disp['p-Wilcoxon'].apply(fmt_pval)
df_disp['Sign.']    = df_disp['Sign.'].apply(fmt_sig)
df_disp['Δ MAE %']  = df_disp['Δ MAE %'].apply(lambda x: f'{x:.1f}%')

display(df_disp.style
    .set_caption('Tabela Consolidada de Validação — Todos os Splits')
    .hide(axis='index')
)

## 5. Visualizações

In [ ]:
# ── 5A. Barplot R² — MTL vs Random Forest por split e via ─────

fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
fig.suptitle('R² — MTL vs Random Forest (Single-Task)', fontsize=14, fontweight='bold', y=1.01)

x = np.arange(len(VIAS))
width = 0.35

for ax, split in zip(axes, SPLITS):
    # MTL
    df_s = df_comp[df_comp['Split'] == split]
    mtl = df_s[df_s['Modelo'] == 'MTL (nosso)'].set_index('Via')
    rf  = df_s[df_s['Modelo'] == 'Random Forest (ST)'].set_index('Via')

    vals_mtl = [mtl.loc[v, 'R2'] if v in mtl.index else np.nan for v in VIAS]
    vals_rf  = [rf.loc[v, 'R2']  if v in rf.index  else np.nan for v in VIAS]

    bars_mtl = ax.bar(x - width/2, vals_mtl, width, label='MTL (nosso)',
                      color=CORES_MODELO['MTL (nosso)'], alpha=0.85, edgecolor='white')
    bars_rf  = ax.bar(x + width/2, vals_rf,  width, label='Random Forest (ST)',
                      color=CORES_MODELO['Random Forest (ST)'], alpha=0.85, edgecolor='white')

    ax.set_title(SPLIT_LABEL[split], fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([VIAS_LABEL[v] for v in VIAS], rotation=35, ha='right', fontsize=9)
    ax.set_ylim(0, 0.85)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    if ax == axes[0]:
        ax.set_ylabel('R²', fontsize=11)
    if ax == axes[-1]:
        ax.legend(fontsize=9, loc='upper right')

plt.tight_layout()

# Salvar
out_path = PATHS['plots_validacao'] / 'comparacao_r2_mtl_vs_rf.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Salvo: {out_path}')
plt.show()

In [ ]:
# ── 5B. Heatmap R² — todos os modelos × split/via ─────────────
MODELOS_ORDER = ['MTL (nosso)', 'Random Forest (ST)', 'Ridge Regression', 'Dummy (mean)']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Heatmap R² — Comparação de Modelos', fontsize=14, fontweight='bold')

for ax, split in zip(axes, SPLITS):
    df_s = df_comp[df_comp['Split'] == split]
    pivot = df_s.pivot_table(index='Via', columns='Modelo', values='R2', aggfunc='first')
    cols = [c for c in MODELOS_ORDER if c in pivot.columns]
    pivot = pivot[cols].reindex(VIAS)
    pivot.index = [VIAS_LABEL[v] for v in VIAS]

    vals = pivot.values.astype(float)
    im = ax.imshow(vals, aspect='auto', cmap='RdYlGn', vmin=-0.05, vmax=0.75)

    ax.set_xticks(range(len(cols)))
    ax.set_xticklabels(cols, rotation=30, ha='right', fontsize=9)
    ax.set_yticks(range(len(VIAS)))
    ax.set_yticklabels([VIAS_LABEL[v] for v in VIAS], fontsize=9)
    ax.set_title(SPLIT_LABEL[split], fontsize=12, fontweight='bold')

    # Anotações
    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            v = vals[i, j]
            color = 'black' if 0.1 < v < 0.65 else 'white'
            ax.text(j, i, f'{v:.3f}' if not np.isnan(v) else '—',
                    ha='center', va='center', fontsize=8.5, color=color, fontweight='bold')

fig.colorbar(im, ax=axes[-1], label='R²', fraction=0.03, pad=0.04)
plt.tight_layout()

out_path = PATHS['plots_validacao'] / 'heatmap_r2_todos_modelos.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Salvo: {out_path}')
plt.show()

In [ ]:
# ── 5C. Melhoria % do MTL sobre Dummy baseline por split ─────
fig, ax = plt.subplots(figsize=(11, 5))

x       = np.arange(len(VIAS))
n_split = len(SPLITS)
total_w = 0.7
w       = total_w / n_split
cores   = ['#2196F3', '#4CAF50', '#FF9800']

for i, split in enumerate(SPLITS):
    if split not in stats_mtl:
        continue
    df = stats_mtl[split]
    vals = [float(df.loc[v, 'Melhoria_%']) if v in df.index else 0.0 for v in VIAS]
    offset = (i - n_split/2 + 0.5) * w
    ax.bar(x + offset, vals, width=w * 0.9,
           label=SPLIT_LABEL[split], color=cores[i], alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels([VIAS_LABEL[v] for v in VIAS], fontsize=10)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Redução MAE vs Baseline (%)', fontsize=11)
ax.set_title('Melhoria do MTL sobre Baseline Nulo — por Via e Split', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.tight_layout()
out_path = PATHS['plots_validacao'] / 'melhoria_pct_mtl_vs_baseline.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Salvo: {out_path}')
plt.show()

In [ ]:
# ── 5D. R² com IC95 (barras de erro bootstrap) ────────────────
# Carregar CorrelacaoBootstrap com IC95

fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
fig.suptitle('R² com IC 95% (Bootstrap) — MTL por Split', fontsize=14, fontweight='bold')

for ax, split in zip(axes, SPLITS):
    path = PATHS['tabelas_validacao'] / split / f'Tabela_Estatistica_{split}.xlsx'
    xl   = pd.ExcelFile(path)
    if 'Correlação Bootstrap' not in xl.sheet_names:
        ax.set_title(f'{SPLIT_LABEL[split]} — sem dados')
        continue

    df = xl.parse('Correlação Bootstrap', index_col=0)

    vias_ok = [v for v in VIAS if v in df.index]
    r2_vals, r2_lo, r2_hi = [], [], []

    for via in vias_ok:
        r2_vals.append(float(df.loc[via, 'R2']))
        ic = str(df.loc[via, 'R2_IC95'])
        try:
            lo, hi = ic.strip('[]').split(';')
            r2_lo.append(float(lo))
            r2_hi.append(float(hi))
        except:
            r2_lo.append(np.nan)
            r2_hi.append(np.nan)

    r2_arr = np.array(r2_vals)
    lo_arr = np.array(r2_lo)
    hi_arr = np.array(r2_hi)
    x_pos  = np.arange(len(vias_ok))

    ax.bar(x_pos, r2_arr, color='#2196F3', alpha=0.8, edgecolor='white')
    ax.errorbar(x_pos, r2_arr,
                yerr=[r2_arr - lo_arr, hi_arr - r2_arr],
                fmt='none', color='#0D47A1', capsize=5, linewidth=1.5)

    ax.set_xticks(x_pos)
    ax.set_xticklabels([VIAS_LABEL[v] for v in vias_ok], rotation=35, ha='right', fontsize=9)
    ax.set_title(SPLIT_LABEL[split], fontsize=12, fontweight='bold')
    ax.set_ylim(0, 0.85)
    if ax == axes[0]:
        ax.set_ylabel('R²', fontsize=11)

plt.tight_layout()
out_path = PATHS['plots_validacao'] / 'r2_ic95_bootstrap_mtl.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Salvo: {out_path}')
plt.show()

## 6. Diagnóstico — O que está faltando nas Tabelas Estatísticas

Os arquivos `Tabela_Estatistica_*.xlsx` gerados até agora **não possuem** as sheets:
- `Y-Scrambling` — testes de validação contra correlação espúria
- `Williams Plot` — domínio de aplicabilidade (leverage vs resíduo)

Essas etapas requerem re-executar o modelo com GPU. As células abaixo só checam o que foi gerado.

In [ ]:
# ── Diagnóstico: sheets presentes vs esperadas ─────────────────
SHEETS_ESPERADAS = [
    'Resumo_Publicacao_ABNT',
    'Correlação Bootstrap',
    'Wilcoxon Signed-Rank',
    'Williams Plot',
    'Y-Scrambling',
]

print(f"{'Split':12s}  {'Sheet':35s}  Status")
print('─' * 60)

for split in SPLITS:
    path = PATHS['tabelas_validacao'] / split / f'Tabela_Estatistica_{split}.xlsx'
    xl = pd.ExcelFile(path)
    for s in SHEETS_ESPERADAS:
        ok = '✓' if s in xl.sheet_names else '✗  AUSENTE'
        print(f'{SPLIT_LABEL[split]:12s}  {s:35s}  {ok}')
    print()

## 7. Exportar Tabela Consolidada para Excel

In [ ]:
# Exportar tabela de comparação completa
out_excel = PATHS['tabelas_validacao'] / 'Comparacao_MTL_vs_Benchmarks.xlsx'

with pd.ExcelWriter(out_excel, engine='openpyxl') as writer:

    # Aba 1: Comparação R² e MAE
    df_comp.to_excel(writer, sheet_name='Comparacao_Geral', index=False)

    # Aba 2: Publicação consolidada
    df_pub_all.to_excel(writer, sheet_name='Publicacao_Consolidada', index=False)

    # Uma aba por split com pivot R²
    for split in SPLITS:
        df_s = df_comp[df_comp['Split'] == split]
        pivot_r2  = df_s.pivot_table(index='Via', columns='Modelo', values='R2',  aggfunc='first')
        pivot_mae = df_s.pivot_table(index='Via', columns='Modelo', values='MAE', aggfunc='first')

        # Merge R² e MAE
        pivot_r2.columns  = [f'R2_{c}'  for c in pivot_r2.columns]
        pivot_mae.columns = [f'MAE_{c}' for c in pivot_mae.columns]
        merged = pd.concat([pivot_r2, pivot_mae], axis=1)
        merged.to_excel(writer, sheet_name=f'Metricas_{SPLIT_LABEL[split]}')

print(f'Exportado: {out_excel}')